In [1]:
# -*- coding: utf-8 -*-
"""colab_A.py：驗證「只用白底圖訓練（階段 1）的模型，換到真實照片會掉多少」。

做的事（全部自動，你只要執行一格）：
  1. 連接 Google Drive、解壓縮專案和 Fruits-360（跟 Colab_A 的步驟 1～5 一樣）
  2. 找三個模型「階段 1」的權重（a_*_stage1.pth）。Drive 的 fruit_AB_out 有就直接用；
     沒有就用 A3_train.py --skip_stage2 只訓練階段 1（每個模型約 5～6 分鐘，種子固定 42）
  3. 用專案原本的評估函式，在 real_test109 上測「階段 1」和「最終」（加了真實照片）兩個版本
  4. 印出對照表，存成 Drive 的 fruit_AB_out/stage1_vs_final.csv

用法：在 Colab 開一個新筆記本，執行階段選 T4 GPU，把這整個檔案貼進一格，按 ▶。
Drive 最上層要有 fruit_AB.zip 和 archive.zip（Fruits-360）。
"""
import csv
import os
import subprocess
import sys

DRIVE = "/content/drive/MyDrive"
OUT = DRIVE + "/fruit_AB_out"          # Colab_A 把權重和成績存在這裡
PROJ = "/content/fruit_AB"
MODELS = ["resnet18", "mobilenetv2", "baseline"]
EPOCHS1 = 15                            # 跟 Colab_A 一樣
STAGE1_DIR = PROJ + "/models_stage1"


def sh(cmd):
    """執行一行指令，即時把輸出印在畫面上；失敗就停下來。"""
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError("指令失敗（代碼 %d）：%s" % (p.returncode, cmd))


def pct(x):
    return "—" if x is None else "%.1f%%" % (100.0 * x)


def make_table(rows):
    """rows：[(模型, 版本, 白底original Top1, 真實單一Top1, 真實單一Top3, 單一張數), ...] -> 文字表格"""
    head = ["模型", "版本", "白底 original Top1", "真實照片 Top1", "真實照片 Top3", "單一水果張數"]
    lines = [head]
    for m, v, w, t1, t3, n in rows:
        lines.append([m, v, pct(w), pct(t1), pct(t3), str(n)])
    widths = [max(len(r[i]) for r in lines) for i in range(len(head))]
    out = []
    for k, r in enumerate(lines):
        out.append("  ".join(r[i].ljust(widths[i]) for i in range(len(head))))
        if k == 0:
            out.append("  ".join("-" * widths[i] for i in range(len(head))))
    return "\n".join(out)


def prepare():
    # ---- 1. Drive、專案、資料 ----
    if not os.path.isdir(DRIVE):
        from google.colab import drive
        drive.mount("/content/drive")
    if not os.path.isdir(PROJ):
        sh("cp %s/fruit_AB.zip /content/ && unzip -q -o /content/fruit_AB.zip -d /content/" % DRIVE)
    os.chdir(PROJ)
    sh("pip -q install pyyaml")
    if PROJ not in sys.path:
        sys.path.insert(0, PROJ)

    real_in_drive = os.path.isdir(OUT + "/real")
    if not os.path.isdir("/content/data/fruits-360_100x100"):
        folders = '"fruits-360_100x100/*" "fruits-360_original-size/*"'
        if not real_in_drive:
            folders += ' "fruits-360_multi/*"'
        print("解壓縮 Fruits-360（會花幾分鐘）…")
        sh("unzip -q -o %s/archive.zip %s -d /content/data" % (DRIVE, folders))

    # ---- 2. 真實照片：優先用 Colab_A 當初存在 Drive 的那一份，切法才會完全一樣 ----
    if not os.path.isdir("/content/real"):
        if real_in_drive:
            sh("cp -r %s/real /content/real" % OUT)
        else:
            print("Drive 找不到 fruit_AB_out/real，改用 A1 重新整理（切法固定，應與原本相同）")
            sh("python A/A1_prepare_real_photos.py --src /content/data/fruits-360_multi/test-multiple_fruits --out /content/real")

    # ---- 3. 清單 ----
    if not os.path.exists("/content/lists/real_lists.json"):
        sh("python A/A2_build_lists.py --fruits100 /content/data/fruits-360_100x100/fruits-360 "
           "--original /content/data/fruits-360_original-size/fruits-360-original-size "
           "--real_dir /content/real --out /content/lists")


def find_or_train_stage1(model):
    """回傳階段 1 權重的路徑：Drive 有就用；沒有就只訓練階段 1。"""
    for path in (OUT + "/a_%s_stage1.pth" % model, STAGE1_DIR + "/a_%s_stage1.pth" % model):
        if os.path.exists(path):
            print("找到階段 1 權重：", path)
            return path
    print("Drive 沒有 %s 的階段 1 權重，現在只訓練階段 1（約 5～6 分鐘）…" % model)
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("沒有 GPU，訓練會非常慢。請在 Colab：執行階段 → 變更執行階段類型 → T4 GPU，再重跑。")
    sh("python A/A3_train.py --model %s --lists /content/lists --white_source both --epochs1 %d "
       "--skip_stage2 --save_dir %s" % (model, EPOCHS1, STAGE1_DIR))
    path = STAGE1_DIR + "/a_%s_stage1.pth" % model
    sh("mkdir -p %s && cp %s %s/" % (OUT, path, OUT))   # 存一份回 Drive，下次不用重訓
    return path


def evaluate_checkpoint(path, white, real, device):
    import torch
    import a_common as ac
    import a_data_utils as du
    ck = torch.load(path, map_location=device)
    families = ck["families"]
    n = len(families)
    model = ac.build_model(ck["model_name"], n, pretrained=False)
    model.load_state_dict(ck["state_dict"])
    model.to(device)
    model.eval()
    w = du.evaluate_white(model, white.get("original_test", []), n, device, 64, 2, 5000, 42)
    r = du.evaluate_real(model, real["test"], n, device, 64, 2)
    return w.get("top1"), r.get("single_top1"), r.get("single_top3"), r.get("single_n", 0)


def main():
    import json
    import torch
    prepare()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    with open("/content/lists/white_lists.json", "r", encoding="utf-8") as f:
        white = json.load(f)
    with open("/content/lists/real_lists.json", "r", encoding="utf-8") as f:
        real = json.load(f)
    print("真實照片 test：%d 張（應為 109）" % len(real.get("test", [])))

    rows = []
    for m in MODELS:
        stage1 = find_or_train_stage1(m)
        rows.append((m,) + ("階段 1（只有白底圖）",) + evaluate_checkpoint(stage1, white, real, device))
        final = OUT + "/a_%s.pth" % m
        if os.path.exists(final):
            rows.append((m,) + ("最終（加了真實照片）",) + evaluate_checkpoint(final, white, real, device))
        else:
            print("Drive 找不到最終權重 %s，只顯示階段 1。" % final)

    print("\n" + "=" * 70)
    print(make_table(rows))
    print("=" * 70)
    for m in MODELS:
        s = [r for r in rows if r[0] == m and r[1].startswith("階段 1")]
        f = [r for r in rows if r[0] == m and r[1].startswith("最終")]
        if s and f and s[0][3] is not None and f[0][3] is not None:
            print("%s：真實照片 Top1 %s → %s（加入真實照片後 %+.1f 個百分點）" %
                  (m, pct(s[0][3]), pct(f[0][3]), 100.0 * (f[0][3] - s[0][3])))
    if rows and rows[0][5] != 97:
        print("提醒：真實單一水果張數是 %s，不是 97，切分可能和原本不同，請先別把數字放進簡報。" % rows[0][5])

    os.makedirs(OUT, exist_ok=True)
    out_csv = OUT + "/stage1_vs_final.csv"
    with open(out_csv, "w", newline="", encoding="utf-8-sig") as fh:
        wr = csv.writer(fh)
        wr.writerow(["模型", "版本", "白底original_Top1", "真實單一Top1", "真實單一Top3", "單一水果張數"])
        for r in rows:
            wr.writerow(list(r))
    print("已存到", out_csv)


if __name__ == "__main__":
    main()


Mounted at /content/drive
$ cp /content/drive/MyDrive/fruit_AB.zip /content/ && unzip -q -o /content/fruit_AB.zip -d /content/
$ pip -q install pyyaml
解壓縮 Fruits-360（會花幾分鐘）…
$ unzip -q -o /content/drive/MyDrive/archive.zip "fruits-360_100x100/*" "fruits-360_original-size/*" -d /content/data
$ cp -r /content/drive/MyDrive/fruit_AB_out/real /content/real
$ python A/A2_build_lists.py --fruits100 /content/data/fruits-360_100x100/fruits-360 --original /content/data/fruits-360_original-size/fruits-360-original-size --real_dir /content/real --out /content/lists
掃描 100 Training : 268 個類別
掃描 100 Test : 268 個類別
掃描 original Training : 152 個類別
掃描 original Validation : 152 個類別
掃描 original Test : 152 個類別
水果大類共 55 種
  清單 100_training : 99966 張
  清單 100_test : 33329 張
  清單 original_training : 37911 張
  清單 original_validation : 18960 張
  清單 original_test : 18855 張
真實照片：train 528 / val 101 / test 109 ；因為種類不在 Fruits-360 大類而略過 1 張
  檔名中不在 Fruits-360 水果大類的字（已忽略）： ['kachi', 'tamarilo']

大類   白底100訓練  白底orig